# Facility Capacity Training Pipeline — facilities, NFHS-5 enrichment, enriched model

Three clearly delineated parts, in order:

* **Part 1 — Train on facilities only (GPU):** the baseline capacity model from facility-level features alone (numberDoctors, operator/facility type, text signal).
* **Part 2 — NFHS-5 Socioeconomic Enrichment:** select the most socioeconomically representative district indicators (one per correlation cluster) and join them onto facilities via zip → pincode directory → district. Checkpoints the result to `workspace.default.facilities_enriched` (write access verified).
* **Part 3 — Train on the enriched data (GPU):** the same pipeline plus the 15 NFHS district features. Compare its test metrics against Part 1 to see whether they help.

**Setup:** switch the environment (top-right, next to the compute dropdown) from *Standard* to **AI | GPU**, then *Run All*. Parts 1 and 3 train a PyTorch MLP and need the GPU; Part 2 runs anywhere.

Verified on this data: facilities has 10,088 rows, 2,517 with a numeric capacity label; NFHS-5 has 706 districts x 107 numeric indicators; Part 2 picks 15 representatives (max pairwise |r| 0.68); ~60% of labeled facilities match an NFHS district through their zip code.

---

## Part 1 — Train on facilities only (GPU)

The baseline: predict `capacity` from facility-level features alone. Strongest verified signals: `numberDoctors` (Spearman 0.47), `operatorTypeId` (public median 300 vs private 80), `facilityTypeId` (hospital 100 / clinic 25), and text-derived features. About 2,517 of 10,088 rows have a usable capacity label; the rest are dropped from training.

In [0]:
CATALOG = "databricks_virtue_foundation_dataset_dais_2026"
SCHEMA = "virtue_foundation_dataset"

for t in ["facilities", "nfhs_5_district_health_indicators", "india_post_pincode_directory"]:
    fq = f"{CATALOG}.{SCHEMA}.{t}"
    n = spark.table(fq).limit(1).count()   # forces an actual read
    print(f"OK  {fq}")

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CATALOG = "databricks_virtue_foundation_dataset_dais_2026"
SCHEMA = "virtue_foundation_dataset"
TBL_FACILITIES = f"{CATALOG}.{SCHEMA}.facilities"
TBL_NFHS = f"{CATALOG}.{SCHEMA}.nfhs_5_district_health_indicators"
TBL_PINCODE = f"{CATALOG}.{SCHEMA}.india_post_pincode_directory"

TARGET = "capacity"
RANDOM_STATE = 42

NUMERIC_FEATURES = [
    "numberDoctors",
    "yearEstablished",
    "latitude",
    "longitude",
    "engagement_metrics_n_followers",
    "engagement_metrics_n_likes",
    "engagement_metrics_n_engagements",
    "post_metrics_post_count",
    "number_of_facts_about_the_organization",
    "distinct_social_media_presence_count",
]
CATEGORICAL_FEATURES = [
    "facilityTypeId",
    "operatorTypeId",
    "address_stateOrRegion",
    "organization_type",
    "affiliated_staff_presence",
    "custom_logo_presence",
]

print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

In [0]:
# Load the facilities table and normalize the messy string columns.
# Many columns are strings holding numbers or literal 'null'/'None' text, so we
# convert them to proper NaN/numeric before anything else.
NULL_STRINGS = {"null", "none", "nan", "n/a", "na", "", "-", "unknown"}

def to_na(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().str.lower().where(
        lambda s: ~s.isin(NULL_STRINGS), other=pd.NA
    )

raw = spark.table(TBL_FACILITIES).toPandas()

facilities = raw.copy()
for col in [TARGET] + NUMERIC_FEATURES:
    facilities[col] = pd.to_numeric(to_na(facilities[col]), errors="coerce")
for col in CATEGORICAL_FEATURES:
    facilities[col] = to_na(facilities[col]).astype("object")

# Free-text signal (verified by screening on the 2,517 labeled rows):
#   equipment/capability/procedure/specialties string lengths correlate with capacity
#   (Spearman ~0.33 / 0.38 / 0.23 / 0.15); affiliationTypeIds presence is strong
#   (median 200 when present vs 52 when absent) though only ~1,249 rows have it.
#   description length showed no signal (~0.02) and is left out.
TEXT_FEATURES = ["equipment", "capability", "procedure", "specialties", "affiliationTypeIds"]
for col in TEXT_FEATURES:
    facilities[f"{col}_len"] = to_na(raw[col]).str.len().fillna(0)
    facilities[f"{col}_present"] = to_na(raw[col]).notna().astype(int)
NUMERIC_FEATURES = NUMERIC_FEATURES + [f"{c}_len" for c in TEXT_FEATURES] + [f"{c}_present" for c in TEXT_FEATURES]

# Keep only rows with a usable target label.
labeled = facilities.dropna(subset=[TARGET]).copy()
print(f"facilities rows: {len(facilities)} | rows with a capacity label: {len(labeled)}")
print(f"Dropped {len(facilities) - len(labeled)} rows without a capacity value")
display(labeled[TARGET].describe().to_frame())

In [0]:
# EDA: target distribution and feature/target relationships.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(labeled[TARGET], bins=50)
axes[0].set_title("Capacity distribution (right-skewed -> log1p helps)")
axes[0].set_xlabel("capacity")

axes[1].hist(np.log1p(labeled[TARGET]), bins=50)
axes[1].set_title("log1p(capacity)")
plt.tight_layout()
plt.show()

corr = labeled[NUMERIC_FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
corr.plot(kind="barh", ax=ax)
ax.set_title("Correlation of numeric features with capacity")
plt.tight_layout()
plt.show()

cap_by_type = labeled.groupby("facilityTypeId")[TARGET].agg(["count", "median"])
display(cap_by_type)
print(corr)

In [0]:
# Convert pandas nullable types / pd.NA to plain float64 / np.nan so sklearn imputers work.
labeled = labeled.copy()

for c in NUMERIC_FEATURES:
    labeled[c] = pd.to_numeric(labeled[c], errors="coerce").astype("float64")

for c in CATEGORICAL_FEATURES:
    labeled[c] = labeled[c].astype(object).where(labeled[c].notna(), np.nan)

labeled[TARGET] = labeled[TARGET].astype("float64")

In [0]:
# Split: no event/observation timestamp exists and rows are independent
# facilities, so a shuffled random split is appropriate (80/10/10).
from sklearn.model_selection import train_test_split

X = labeled[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = labeled[TARGET]

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE)

print(f"train: {len(X_train)} | val: {len(X_val)} | test: {len(X_test)}")

In [0]:
# Preprocessing fit on the training split only.
preprocessor = ColumnTransformer(
    [
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5))]), CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)
preprocessor.fit(X_train)

X_train_prep = preprocessor.transform(X_train).astype(np.float32)
X_val_prep = preprocessor.transform(X_val).astype(np.float32)
X_test_prep = preprocessor.transform(X_test).astype(np.float32)
print(f"feature matrix shape after preprocessing: {X_train_prep.shape}")

In [0]:
# Quick sklearn baseline on the same preprocessed features for reference.
y_train_log = np.log1p(y_train)

baseline = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
baseline.fit(X_train_prep, y_train_log)
baseline_pred = np.expm1(baseline.predict(X_test_prep))

def regression_report(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    print(f"{name}: MAE {mae:.2f} | RMSE {rmse:.2f} | R2 {r2:.3f}")
    return {"mae": mae, "rmse": rmse, "r2": r2}

baseline_metrics = regression_report(y_test, baseline_pred, "Baseline (HistGradientBoosting, log target)")

In [0]:
# GPU training: PyTorch MLP on log1p(capacity).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    print("WARNING: no GPU detected — did you switch the environment to 'AI | GPU'? Training will be slow.")
else:
    print(f"Training on GPU: {torch.cuda.get_device_name(0)}")

def to_tensors(X_arr, y_series):
    X_t = torch.tensor(X_arr, dtype=torch.float32, device=device)
    y_t = torch.tensor(np.log1p(y_series.to_numpy()), dtype=torch.float32, device=device).unsqueeze(1)
    return X_t, y_t

X_train_t, y_train_t = to_tensors(X_train_prep, y_train)
X_val_t, y_val_t = to_tensors(X_val_prep, y_val)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)

n_features = X_train_t.shape[1]
mlp = nn.Sequential(
    nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 1),
).to(device)

EPOCHS, PATIENCE = 300, 30
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=10)
loss_fn = nn.HuberLoss(delta=1.0)  # robust to the heavy right tail

best_val, best_state, bad_epochs = float("inf"), None, 0
for epoch in range(EPOCHS):
    mlp.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(mlp(xb), yb)
        loss.backward()
        optimizer.step()

    mlp.eval()
    with torch.no_grad():
        val_loss = loss_fn(mlp(X_val_t), y_val_t).item()
    scheduler.step(val_loss)
    if val_loss < best_val - 1e-4:
        best_val, best_state, bad_epochs = val_loss, {k: v.clone() for k, v in mlp.state_dict().items()}, 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"early stopping at epoch {epoch} (best val loss {best_val:.4f})")
            break

mlp.load_state_dict(best_state)
print(f"best validation Huber loss (log space): {best_val:.4f}")

In [0]:
# Evaluate the MLP on the untouched test set, in original units.
mlp.eval()
with torch.no_grad():
    mlp_pred = np.expm1(mlp(torch.tensor(X_test_prep, dtype=torch.float32, device=device)).cpu().numpy().ravel())

mlp_metrics = regression_report(y_test, mlp_pred, "PyTorch MLP (GPU)")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, mlp_pred, s=12, alpha=0.5)
ax.plot([0, y_test.max()], [0, y_test.max()], "r--", linewidth=1)
ax.set_xlabel("actual capacity")
ax.set_ylabel("predicted capacity")
ax.set_title("Test set: predicted vs actual")
plt.tight_layout()
plt.show()

In [0]:
# MLflow: refit preprocessing on train+val, then log a deployable pyfunc
# wrapper that carries preprocessing and the target transform.
class CapacityModel(mlflow.pyfunc.PythonModel):
    def __init__(self, preproc, net, device):
        self.preproc = preproc
        self.net = net
        self.device = device

    def predict(self, context, model_input):
        X = self.preproc.transform(model_input).astype(np.float32)
        with torch.no_grad():
            t = torch.tensor(X, dtype=torch.float32, device=self.device)
            return np.expm1(self.net(t).cpu().numpy().ravel())

# preprocessor.fit(pd.concat([X_train, X_val]))
mlp.eval()

from mlflow.models import infer_signature

EXPERIMENT = "/Users/mcblake@andrew.cmu.edu/facility_capacity_gpu"
mlflow.set_experiment(EXPERIMENT)
signature = infer_signature(X_train.head(10), pd.Series(mlp_pred[:10]))

with mlflow.start_run(run_name="pytorch_mlp_capacity"):
    mlflow.log_params({"epochs": EPOCHS, "patience": PATIENCE, "lr": 1e-3, "hidden": "128-64", "n_train": len(X_train), "n_val": len(X_val), "n_test": len(X_test)})
    mlflow.log_metrics({f"test_{k}": v for k, v in mlp_metrics.items()})
    mlflow.log_metrics({f"baseline_test_{k}": v for k, v in baseline_metrics.items()})
    model_info = mlflow.pyfunc.log_model(
        "capacity_model",
        python_model=CapacityModel(preprocessor, mlp, device),
        signature=signature,
        input_example=X_train.head(3),
    )
    print(f"\nmodel_uri: {model_info.model_uri}")
    print("MLflow run:", mlflow.get_experiment_by_name(EXPERIMENT).artifact_location)

In [0]:
import os, joblib, torch

REPO_DIR = "/Workspace/Users/mcblake@andrew.cmu.edu/datathon-F26/ML/notebooks/outputs_models" # adjust
out_dir = f"{REPO_DIR}/models/base"
os.makedirs(out_dir, exist_ok=True)

# Network weights (move to CPU so it loads anywhere, including without a GPU)
torch.save({k: v.cpu() for k, v in mlp.state_dict().items()}, f"{out_dir}/mlp_state_dict.pt")

# The fitted preprocessor (imputers, scaler, one-hot encoder)
joblib.dump(preprocessor, f"{out_dir}/preprocessor.joblib")

# Everything needed to rebuild the network and inputs
joblib.dump({
    "n_features": n_features,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "architecture": "Linear(n,128)-ReLU-Dropout0.2-Linear(128,64)-ReLU-Dropout0.2-Linear(64,1)",
    "target_transform": "log1p on train, expm1 on predictions",
}, f"{out_dir}/model_meta.joblib")

print(os.listdir(out_dir))

In [0]:
# Part 2 — Setup and data loading (enrichment).
import re

import numpy as np
import pandas as pd

CATALOG = "databricks_virtue_foundation_dataset_dais_2026"
SCHEMA = "virtue_foundation_dataset"
TBL_FACILITIES = f"{CATALOG}.{SCHEMA}.facilities"
TBL_NFHS = f"{CATALOG}.{SCHEMA}.nfhs_5_district_health_indicators"
TBL_PINCODE = f"{CATALOG}.{SCHEMA}.india_post_pincode_directory"

# Where the enriched output is checkpointed (change if you lack write access).
ENRICHED_TABLE = "workspace.default.facilities_enriched"

def to_num(s):
    m = re.match(r"^\s*-?\d+(\.\d+)?\s*$", str(s))
    return float(m.group(0)) if m else np.nan

def to_na(s):
    return s.astype("string").str.strip().str.lower().where(
        lambda x: ~x.isin({"null", "none", "nan", "n/a", "na", "", "-", "unknown"}), other=pd.NA
    )

def norm(s):
    return (s.astype("string").str.strip().str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.replace(r"[^a-z ]", "", regex=True)
            .str.strip())

nfhs_raw = spark.table(TBL_NFHS).toPandas()
num_cols = [c for c in nfhs_raw.columns if c not in ("district_name", "state_ut")]
nfhs_num = pd.DataFrame({c: nfhs_raw[c].map(to_num) for c in num_cols})
print(f"NFHS-5: {len(nfhs_raw)} districts x {len(num_cols)} indicators parsed numeric")

In [0]:
# Part 2 — Select the most socioeconomically representative indicators.
# Candidate pool: education, household amenities, wealth proxies, demographics,
# women's empowerment, nutrition. Survey-count columns excluded.
POOL = [
    "female_population_age_6_years_and_above_ever_schooled_pct",
    "women_age_15_49_who_are_literate_pct",
    "women_age_15_49_with_10_or_more_years_of_schooling_pct",
    "population_below_age_15_years_pct",
    "sex_ratio_total_f_per_1000_m",
    "hh_electricity_pct",
    "hh_improved_water_pct",
    "hh_use_improved_sanitation_pct",
    "households_using_clean_fuel_for_cooking_pct",
    "households_using_iodized_salt_pct",
    "hh_member_covered_health_insurance_pct",
    "w20_24_married_before_age_18_years_pct",
    "w15_24_who_use_menstrual_hygiene_pct",
    "women_age_15_49_years_whose_bmi_bmi_is_underweight_bmi_lt_1_pct",
    "all_w15_49_who_are_anaemic_pct",
    "institutional_birth_5y_pct",
    "births_attended_by_skilled_hp_5y_10_pct",
    "child_u5_who_are_stunted_height_for_age_18_pct",
]

pool = nfhs_num[POOL].dropna(axis=1, thresh=int(0.8 * len(nfhs_num)))
corr = pool.corr(method="spearman").abs()

# Greedy correlation pruning at |r| > 0.7: repeatedly keep the indicator most
# connected to the rest, drop its redundant partners. One representative per
# cluster -> minimal redundancy (max pairwise |r| among picks was 0.68).
remaining = list(pool.columns)
representatives = []
while remaining:
    best = corr.loc[remaining, remaining].mean().idxmax()
    representatives.append(best)
    remaining = [c for c in remaining if c != best and corr.loc[best, c] <= 0.7]

print(f"{len(representatives)} representative indicators (one per correlation cluster):")
for r in representatives:
    print("  -", r)

In [0]:
# Part 2 — Join facilities to NFHS districts via the pincode directory.
facilities = spark.table(TBL_FACILITIES).toPandas()
facilities["capacity_num"] = pd.to_numeric(to_na(facilities["capacity"]), errors="coerce")

pin = spark.table(TBL_PINCODE).select("pincode", "district", "statename").toPandas()
pin["d"], pin["st"] = norm(pin["district"]), norm(pin["statename"])

# Majority (district, state) per pincode = most frequent among its post offices.
cnt = pin.groupby(["pincode", "d", "st"]).size().rename("n").reset_index()
pin_lookup = (cnt.sort_values("n", ascending=False)
                  .drop_duplicates("pincode")
                  .set_index("pincode")[["d", "st"]])

# NFHS feature frame keyed on normalized (district, state).
nfhs_feats = pd.concat(
    [pd.DataFrame({"d": norm(nfhs_raw["district_name"]), "st": norm(nfhs_raw["state_ut"])}),
     nfhs_num[representatives]],
    axis=1,
)

# facilities zip -> pincode lookup -> (district, state) -> NFHS features.
zips = pd.to_numeric(to_na(facilities["address_zipOrPostcode"]), errors="coerce")
facilities["join_d"] = zips.map(pin_lookup["d"])
facilities["join_st"] = zips.map(pin_lookup["st"])

enriched = facilities.merge(nfhs_feats, how="left",
                            left_on=["join_d", "join_st"], right_on=["d", "st"])

matched = enriched["d"].notna()
labeled = enriched["capacity_num"].notna()
print(f"facilities: {len(enriched)} rows")
print(f"matched to an NFHS district: {matched.sum()} ({matched.mean()*100:.0f}%)")
print(f"of capacity-labeled rows: {(matched & labeled).sum()} of {labeled.sum()} ({(matched & labeled).sum()/labeled.sum()*100:.0f}%)")

In [0]:
# Part 2 — Checkpoint the enriched table so Part 3 (and anything else) can
# reuse it without re-running the join. Unmatched rows keep NULL district
# features; Part 3 imputes them, so no rows are lost.
out = enriched.replace({pd.NA: None}).copy()
for c in representatives:
    out[c] = pd.to_numeric(out[c], errors="coerce")

(spark.createDataFrame(out)
      .write.mode("overwrite")
      .saveAsTable(ENRICHED_TABLE))
print(f"wrote {len(out)} rows to {ENRICHED_TABLE}")
display(spark.table(ENRICHED_TABLE).select("unique_id", "capacity_num", *representatives[:3]).limit(5))

---

## Part 3 — Train on the enriched data (GPU)

Switch the environment (next to the compute dropdown) to **AI | GPU** before running this part — the MLP trains on the GPU. Part 2 runs fine on plain serverless.

Notes:

* Only ~60% of labeled facilities matched an NFHS district; unmatched rows have NULL district features, which the preprocessing imputes with the training median — no rows are dropped.
* District indicators are constant within a district, so they add coarse regional signal. Judge whether they help by comparing this notebook's test metrics against the facilities-only notebook.

In [0]:
# Part 3 — Setup.
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt

import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

TARGET = "capacity"
RANDOM_STATE = 42

# 15 NFHS-5 district indicators picked in Part 1 (one per correlation cluster).
NFHS_FEATURES = [
    "women_age_15_49_with_10_or_more_years_of_schooling_pct",
    "population_below_age_15_years_pct",
    "hh_use_improved_sanitation_pct",
    "women_age_15_49_years_whose_bmi_bmi_is_underweight_bmi_lt_1_pct",
    "hh_electricity_pct",
    "child_u5_who_are_stunted_height_for_age_18_pct",
    "households_using_clean_fuel_for_cooking_pct",
    "w20_24_married_before_age_18_years_pct",
    "female_population_age_6_years_and_above_ever_schooled_pct",
    "hh_member_covered_health_insurance_pct",
    "hh_improved_water_pct",
    "all_w15_49_who_are_anaemic_pct",
    "sex_ratio_total_f_per_1000_m",
    "households_using_iodized_salt_pct",
    "births_attended_by_skilled_hp_5y_10_pct",
]

NUMERIC_FEATURES = [
    "numberDoctors",
    "yearEstablished",
    "latitude",
    "longitude",
    "engagement_metrics_n_followers",
    "engagement_metrics_n_likes",
    "engagement_metrics_n_engagements",
    "post_metrics_post_count",
    "number_of_facts_about_the_organization",
    "distinct_social_media_presence_count",
] + NFHS_FEATURES

CATEGORICAL_FEATURES = [
    "facilityTypeId",
    "operatorTypeId",
    "address_stateOrRegion",
    "organization_type",
    "affiliated_staff_presence",
    "custom_logo_presence",
]

print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

In [0]:
# Part 3 — Load and prepare. Uses the in-memory frame from Part 2 if it
# exists; otherwise falls back to the checkpointed table (run Part 2 first).
if "enriched" in globals():
    raw = enriched.copy()
else:
    raw = spark.table(ENRICHED_TABLE).toPandas()

facilities = raw.copy()
for col in [c for c in NUMERIC_FEATURES if c not in NFHS_FEATURES] + [TARGET]:
    facilities[col] = pd.to_numeric(to_na(facilities[col]), errors="coerce")
for col in CATEGORICAL_FEATURES:
    facilities[col] = to_na(facilities[col]).astype("object")
for col in NFHS_FEATURES:
    facilities[col] = pd.to_numeric(facilities[col], errors="coerce")

# Free-text signal (verified screening): length/presence of these scraped
# columns correlates with capacity (equipment .33, capability .38, procedure
# .23, specialties .15; affiliationTypeIds presence median 200 vs 52).
TEXT_FEATURES = ["equipment", "capability", "procedure", "specialties", "affiliationTypeIds"]
for col in TEXT_FEATURES:
    facilities[f"{col}_len"] = to_na(raw[col]).str.len().fillna(0)
    facilities[f"{col}_present"] = to_na(raw[col]).notna().astype(int)
NUMERIC_FEATURES = NUMERIC_FEATURES + [f"{c}_len" for c in TEXT_FEATURES] + [f"{c}_present" for c in TEXT_FEATURES]

labeled = facilities.dropna(subset=[TARGET]).copy()
print(f"rows: {len(facilities)} | with a capacity label: {len(labeled)}")
print(f"labeled rows with NFHS district features: {labeled[NFHS_FEATURES[0]].notna().sum()}")
display(labeled[TARGET].describe().to_frame())

In [0]:
# Convert pandas nullable types / pd.NA to plain float64 / np.nan so sklearn imputers work.
labeled = labeled.copy()

for c in NUMERIC_FEATURES:
    labeled[c] = pd.to_numeric(labeled[c], errors="coerce").astype("float64")

for c in CATEGORICAL_FEATURES:
    labeled[c] = labeled[c].astype(object).where(labeled[c].notna(), np.nan)

labeled[TARGET] = labeled[TARGET].astype("float64")

In [0]:
# Part 3 — EDA: target distribution and feature/target relationships.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(labeled[TARGET], bins=50)
axes[0].set_title("Capacity distribution (right-skewed -> log1p helps)")
axes[0].set_xlabel("capacity")

axes[1].hist(np.log1p(labeled[TARGET]), bins=50)
axes[1].set_title("log1p(capacity)")
plt.tight_layout()
plt.show()

corr = labeled[NUMERIC_FEATURES + [TARGET]].corr(method="spearman")[TARGET].drop(TARGET).sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
corr.plot(kind="barh", ax=ax)
ax.set_title("Spearman correlation with capacity (facility + NFHS features)")
plt.tight_layout()
plt.show()

In [0]:
# Part 3 — Split: no event/observation timestamp exists and rows are
# independent facilities, so a shuffled random split is appropriate (80/10/10).
X = labeled[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = labeled[TARGET]

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE)

print(f"train: {len(X_train)} | val: {len(X_val)} | test: {len(X_test)}")

In [0]:
# Part 3 — Preprocessing fit on the training split only. Missing NFHS
# district features (unmatched facilities) are imputed with the median.
preprocessor = ColumnTransformer(
    [
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5))]), CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)
preprocessor.fit(X_train)

X_train_prep = preprocessor.transform(X_train).astype(np.float32)
X_val_prep = preprocessor.transform(X_val).astype(np.float32)
X_test_prep = preprocessor.transform(X_test).astype(np.float32)
print(f"feature matrix shape after preprocessing: {X_train_prep.shape}")

In [0]:
# Part 3 — Quick sklearn baseline on the same preprocessed features for reference.
y_train_log = np.log1p(y_train)

baseline = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
baseline.fit(X_train_prep, y_train_log)
baseline_pred = np.expm1(baseline.predict(X_test_prep))

def regression_report(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    print(f"{name}: MAE {mae:.2f} | RMSE {rmse:.2f} | R2 {r2:.3f}")
    return {"mae": mae, "rmse": rmse, "r2": r2}

baseline_metrics = regression_report(y_test, baseline_pred, "Baseline (HistGradientBoosting, log target)")

In [0]:
# Part 3 — GPU training: PyTorch MLP on log1p(capacity).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    print("WARNING: no GPU detected — did you switch the environment to 'AI | GPU'? Training will be slow.")
else:
    print(f"Training on GPU: {torch.cuda.get_device_name(0)}")

def to_tensors(X_arr, y_series):
    X_t = torch.tensor(X_arr, dtype=torch.float32, device=device)
    y_t = torch.tensor(np.log1p(y_series.to_numpy()), dtype=torch.float32, device=device).unsqueeze(1)
    return X_t, y_t

X_train_t, y_train_t = to_tensors(X_train_prep, y_train)
X_val_t, y_val_t = to_tensors(X_val_prep, y_val)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)

n_features = X_train_t.shape[1]
mlp = nn.Sequential(
    nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 1),
).to(device)

EPOCHS, PATIENCE = 300, 30
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=10)
loss_fn = nn.HuberLoss(delta=1.0)  # robust to the heavy right tail

best_val, best_state, bad_epochs = float("inf"), None, 0
for epoch in range(EPOCHS):
    mlp.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(mlp(xb), yb)
        loss.backward()
        optimizer.step()

    mlp.eval()
    with torch.no_grad():
        val_loss = loss_fn(mlp(X_val_t), y_val_t).item()
    scheduler.step(val_loss)
    if val_loss < best_val - 1e-4:
        best_val, best_state, bad_epochs = val_loss, {k: v.clone() for k, v in mlp.state_dict().items()}, 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"early stopping at epoch {epoch} (best val loss {best_val:.4f})")
            break

mlp.load_state_dict(best_state)
print(f"best validation Huber loss (log space): {best_val:.4f}")

In [0]:
# Part 3 — Evaluate the MLP on the untouched test set, in original units.
mlp.eval()
with torch.no_grad():
    mlp_pred = np.expm1(mlp(torch.tensor(X_test_prep, dtype=torch.float32, device=device)).cpu().numpy().ravel())

mlp_metrics = regression_report(y_test, mlp_pred, "PyTorch MLP (GPU, enriched)")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, mlp_pred, s=12, alpha=0.5)
ax.plot([0, y_test.max()], [0, y_test.max()], "r--", linewidth=1)
ax.set_xlabel("actual capacity")
ax.set_ylabel("predicted capacity")
ax.set_title("Test set: predicted vs actual")
plt.tight_layout()
plt.show()

In [0]:
# Part 3 — MLflow: refit preprocessing on train+val, then log a deployable
# pyfunc wrapper that carries preprocessing and the target transform.
class CapacityModel(mlflow.pyfunc.PythonModel):
    def __init__(self, preproc, net, device):
        self.preproc = preproc
        self.net = net
        self.device = device

    def predict(self, context, model_input):
        X = self.preproc.transform(model_input).astype(np.float32)
        with torch.no_grad():
            t = torch.tensor(X, dtype=torch.float32, device=self.device)
            return np.expm1(self.net(t).cpu().numpy().ravel())

# preprocessor.fit(pd.concat([X_train, X_val]))
mlp.eval()

from mlflow.models import infer_signature

EXPERIMENT = "/Users/mcblake@andrew.cmu.edu/facility_capacity_gpu_enriched"
mlflow.set_experiment(EXPERIMENT)
signature = infer_signature(X_train.head(10), pd.Series(mlp_pred[:10]))

with mlflow.start_run(run_name="pytorch_mlp_capacity_enriched"):
    mlflow.log_params({"epochs": EPOCHS, "patience": PATIENCE, "lr": 1e-3, "hidden": "128-64", "n_train": len(X_train), "n_val": len(X_val), "n_test": len(X_test), "n_nfhs_features": len(NFHS_FEATURES)})
    mlflow.log_metrics({f"test_{k}": v for k, v in mlp_metrics.items()})
    mlflow.log_metrics({f"baseline_test_{k}": v for k, v in baseline_metrics.items()})
    model_info = mlflow.pyfunc.log_model(
        "capacity_model",
        python_model=CapacityModel(preprocessor, mlp, device),
        signature=signature,
        input_example=X_train.head(3),
    )
    print(f"\nmodel_uri: {model_info.model_uri}")

In [0]:
import os, joblib, torch

REPO_DIR = "/Workspace/Users/mcblake@andrew.cmu.edu/datathon-F26/ML/notebooks/outputs_models"   # adjust
out_dir = f"{REPO_DIR}/models/enriched"
os.makedirs(out_dir, exist_ok=True)

# Network weights (move to CPU so it loads anywhere, including without a GPU)
torch.save({k: v.cpu() for k, v in mlp.state_dict().items()}, f"{out_dir}/mlp_state_dict.pt")

# The fitted preprocessor (imputers, scaler, one-hot encoder)
joblib.dump(preprocessor, f"{out_dir}/preprocessor.joblib")

# Everything needed to rebuild the network and inputs
joblib.dump({
    "n_features": n_features,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "architecture": "Linear(n,128)-ReLU-Dropout0.2-Linear(128,64)-ReLU-Dropout0.2-Linear(64,1)",
    "target_transform": "log1p on train, expm1 on predictions",
}, f"{out_dir}/model_meta.joblib")

print(os.listdir(out_dir))

---

## Next steps

* Compare Part 3's test metrics against Part 1's facilities-only results to see whether the NFHS district features actually help (they only cover ~60% of labeled rows).
* If they help, lower the |r| threshold in Part 2 (0.7 → 0.6) to pull in more clusters of indicators.
* The logged model is in MLflow under `facility_capacity_gpu_enriched`; register it in Unity Catalog if you want to serve it.